In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

ROOT = Path.cwd()
if not (ROOT / 'artifacts').exists():
    ROOT = Path('c:/PathSense_Complete/PathSense_Complete/PDS/pathsense/ml')

ARTIFACTS = ROOT / 'artifacts'
DATA_PATH = ROOT.parent / 'dataset' / 'pedestrian_accidents.csv'

print('ML root:', ROOT)
print('Dataset path:', DATA_PATH)

In [ ]:
# Equivalent notebook cell for train.py -> load_df()
def load_df(path=DATA_PATH):
    df = pd.read_csv(path)
    df = df.dropna()
    df.columns = df.columns.str.strip()
    return df

df = load_df()
print('Shape:', df.shape)
df.head(5)

In [ ]:
# Equivalent notebook cell for train.py -> categorize_time() and build_xy()
def categorize_time(time_str):
    hour = int(str(time_str).split(':')[0])
    if 5 <= hour < 12:
        return 'Morning'
    if 12 <= hour < 17:
        return 'Afternoon'
    if 17 <= hour < 21:
        return 'Evening'
    return 'Night'

def build_xy(df):
    df = df.copy()
    df['Time_Category'] = df['Time of Day'].apply(categorize_time)
    df = df.drop(columns=['Time of Day'])
    df['High_Risk'] = df['Accident Severity'].apply(lambda x: 1 if x in ['Serious', 'Fatal'] else 0)
    y = df['High_Risk']
    X = df.drop(columns=['Accident Severity', 'High_Risk', 'Pedestrian_Involved'])
    return X, y, df

X_raw, y, working_df = build_xy(df)
print('Feature columns:')
print(list(X_raw.columns))
print('Target distribution:')
print(y.value_counts())

In [ ]:
# Equivalent notebook cell for train.py -> encode_features()
def encode_features(X):
    X = X.copy()
    label_encoders = {}
    for column in X.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X[column] = le.fit_transform(X[column].astype(str))
        label_encoders[column] = le
    return X, label_encoders

X_encoded, label_encoders = encode_features(X_raw)
print('Encoded columns:', list(label_encoders.keys()))
X_encoded.head(5)

In [ ]:
# Equivalent notebook cell for train.py -> train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

In [ ]:
# Equivalent notebook cell for train.py -> model training and comparison
pos = int((y_train == 1).sum())
neg = int((y_train == 0).sum())
scale_pos_weight = float(neg / pos) if pos else 1.0

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    tree_method='hist',
    n_jobs=1,
)
lr = LogisticRegression(max_iter=2000, random_state=42)
dt = DecisionTreeClassifier(max_depth=12, random_state=42)
rf = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=1)

xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
lr.fit(X_train, y_train)
dt.fit(X_train, y_train)
rf.fit(X_train, y_train)

performance_rows = []
for name, model in [('XGBoost', xgb), ('Logistic Regression', lr), ('Decision Tree', dt), ('Random Forest', rf)]:
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    performance_rows.append({
        'Model': name,
        'Accuracy': round(float(accuracy_score(y_test, y_pred)), 4),
        'ROC-AUC': round(float(roc_auc_score(y_test, y_prob)), 4),
    })

performance_df = pd.DataFrame(performance_rows).sort_values('Accuracy', ascending=False)
performance_df

In [ ]:
# Equivalent notebook cell for train.py -> reports and confusion matrix
y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]
print('XGBoost Accuracy:', accuracy_score(y_test, y_pred_xgb))
print('XGBoost ROC-AUC:', roc_auc_score(y_test, y_prob_xgb))
print('\nClassification Report:\n')
print(classification_report(y_test, y_pred_xgb))
print('\nConfusion Matrix:\n')
print(confusion_matrix(y_test, y_pred_xgb))

In [ ]:
# Equivalent notebook cell for train.py -> cross validation
xgb_cv = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    tree_method='hist',
    n_jobs=1,
)
cv_scores = cross_val_score(xgb_cv, X_encoded, y, cv=5, scoring='accuracy')
print('Cross-validation scores:', cv_scores)
print('Average CV Accuracy:', cv_scores.mean())

In [ ]:
# Equivalent notebook cell for predictor.py -> risk bucket and predictor class
def risk_bucket(prob):
    if prob > 0.7:
        return 'VERY_HIGH'
    if prob > 0.4:
        return 'MODERATE'
    return 'LOW'

class DemoRiskPredictor:
    def __init__(self, artifacts_dir=ARTIFACTS):
        self.model = joblib.load(artifacts_dir / 'xgboost_risk.pkl')
        self.label_encoders = joblib.load(artifacts_dir / 'label_encoders.pkl')
        meta = json.loads((artifacts_dir / 'model_meta.json').read_text(encoding='utf-8'))
        self.feature_columns = meta['feature_columns']

    def predict_proba_high_risk(self, input_data):
        sample_df = pd.DataFrame([input_data])
        for column in sample_df.columns:
            if column in self.label_encoders:
                le = self.label_encoders[column]
                val = str(sample_df[column].iloc[0])
                if val not in le.classes_:
                    val = le.classes_[0]
                sample_df[column] = le.transform([val])
        sample_df = sample_df[self.feature_columns]
        return float(self.model.predict_proba(sample_df)[0, 1])

    def alert_message(self, prob):
        bucket = risk_bucket(prob)
        if bucket == 'VERY_HIGH':
            return 'Very high risk of serious or fatal conditions on this segment. Stop and replan if possible.'
        if bucket == 'MODERATE':
            return 'Moderate risk. Proceed with extra caution and stay on safest path.'
        return 'Low estimated risk for serious or fatal accident under these conditions.'

demo_predictor = DemoRiskPredictor()
print('Predictor loaded successfully')

In [ ]:
# Equivalent notebook cell for predictor.py -> demo scenarios
demo_scenarios = {
    'Low': {
        'Day of Week': 'Tuesday',
        'Weather Conditions': 'Clear',
        'Lighting Conditions': 'Daylight',
        'Road Type': 'Urban Road',
        'Road Condition': 'Dry',
        'Traffic Control Presence': 'Signals',
        'Speed Limit (km/h)': 40,
        'Number of Vehicles Involved': 1,
        'Time_Category': 'Morning'
    },
    'High': {
        'Day of Week': 'Saturday',
        'Weather Conditions': 'Stormy',
        'Lighting Conditions': 'Dark',
        'Road Type': 'National Highway',
        'Road Condition': 'Wet',
        'Traffic Control Presence': 'Unknown',
        'Speed Limit (km/h)': 100,
        'Number of Vehicles Involved': 4,
        'Time_Category': 'Night'
    }
}

rows = []
for label, scenario in demo_scenarios.items():
    prob = demo_predictor.predict_proba_high_risk(scenario)
    rows.append({
        'Scenario': label,
        'Probability': round(prob, 4),
        'Risk Bucket': risk_bucket(prob),
        'Message': demo_predictor.alert_message(prob)
    })

pd.DataFrame(rows)

In [ ]:
# Equivalent notebook cell for report_assets.py -> live EDA example 1
working_df['High_Risk_Label'] = working_df['High_Risk'].map({0: 'Low Risk', 1: 'High Risk'})
working_df['High_Risk_Label'].value_counts().reindex(['Low Risk', 'High Risk']).plot(
    kind='bar', color=['#22c55e', '#ef4444'], title='Risk Class Distribution'
)
plt.xlabel('Risk Level')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Equivalent notebook cell for report_assets.py -> live EDA example 2
weather_risk = working_df.groupby(['Weather Conditions', 'High_Risk_Label']).size().unstack(fill_value=0)
weather_risk.plot(kind='bar', color=['#22c55e', '#ef4444'], title='Weather vs Risk', figsize=(9, 5))
plt.xlabel('Weather Conditions')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.show()

In [ ]:
# Equivalent notebook cell for report_assets.py -> use generated performance and observations
report_tables = ROOT / 'artifacts' / 'report_assets' / 'tables'
performance_table = pd.read_csv(report_tables / 'model_performance_table.csv')
comparative_table = pd.read_csv(report_tables / 'comparative_table.csv')
performance_table

In [ ]:
comparative_table

In [ ]:
# Equivalent notebook cell for pds.py -> entry point idea
# pds.py is only a tiny wrapper that calls train.main().
print('pds.py behavior: it simply runs the training pipeline entry point.')
print('Equivalent logic: from train import main; main()')